### DOB ECB Violations - Illegal Conversions

In [49]:
import pandas as pd
pathdob = "...DOB\\"

de = pd.read_csv(pathdob + "DOB_ECB_Violations.csv", low_memory=False)

In [50]:
de["VIOLATION_TYPE"] = de["VIOLATION_TYPE"].astype(str)
de["VIOLATION_DESCRIPTION"] = de["VIOLATION_DESCRIPTION"].astype(str)
#de["VIOLATION_DESCRIPTION"] = de["VIOLATION_DESCRIPTION"].astype(str)

#dv["VIOLATION_NUMBER"] = dv["VIOLATION_NUMBER"].astype(str)
#dv["VIOLATION_TYPE"] = dv["VIOLATION_TYPE"].astype(str)
#dv["VIOLATION_CATEGORY"] = dv["VIOLATION_CATEGORY"].astype(str)

c = sorted(list(set(de["VIOLATION_DESCRIPTION"].tolist() )))

#c

Filter for illegal conversions

In [51]:
print (len(de))

#de2 = de[de.VIOLATION_DESCRIPTION.str.contains("CONV") == True]

de2 = de[
    de['VIOLATION_DESCRIPTION'].str.contains('ILLEG', case=False, na=False) &
    de['VIOLATION_DESCRIPTION'].str.contains('CONV', case=False, na=False)
]

print (len(de2))


1808463
20667


In [52]:
de2.to_csv(pathdob + "test.csv", index=False)

In [53]:
pathdob

'C:\\Users\\MehriD01\\OneDrive - New York City Housing Authority\\Documents\\DOB\\'

In [54]:
de2["VIOLATION_TYPE"].value_counts()

VIOLATION_TYPE
Construction           10771
Quality of Life         8253
Unknown                  924
HPD                      350
Zoning                   261
Signs                     43
Plumbing                  29
Public Assembly           17
Administrative             9
Elevators                  7
Local Law                  1
Boilers                    1
Cranes and Derricks        1
Name: count, dtype: int64

Create BBL

In [55]:
#de2['BLOCK'] = de2['BLOCK'].astype(int).astype(str)
de2 = de2.dropna(subset=['BLOCK'])
# Create BBL
de2['BBL'] = (
    de2['BORO'].astype(int).astype(str) +
    de2['BLOCK'].astype(int).astype(str).str.zfill(5) +
    de2['LOT'].astype(int).astype(str).str.zfill(4)
)

de2.BBL = de2.BBL.astype(str)

In [56]:
subset = ['ECB_VIOLATION_NUMBER', 'ECB_VIOLATION_STATUS', 'VIOLATION_DESCRIPTION', 
       'DOB_VIOLATION_NUMBER','BBL', 'BIN', 'BORO', 'BLOCK', 'LOT', 'HEARING_DATE',
       'HEARING_TIME', 'SERVED_DATE', 'ISSUE_DATE', 'SEVERITY',
       'VIOLATION_TYPE', 'RESPONDENT_NAME', 'RESPONDENT_HOUSE_NUMBER',
       'RESPONDENT_STREET', 'RESPONDENT_CITY', 'RESPONDENT_ZIP',
       'PENALITY_IMPOSED', 'AMOUNT_PAID',
       'BALANCE_DUE', 'AGGRAVATED_LEVEL', 'HEARING_STATUS',
       'CERTIFICATION_STATUS']
de2 = de2[subset]

In [57]:
de2.dtypes

ECB_VIOLATION_NUMBER        object
ECB_VIOLATION_STATUS        object
VIOLATION_DESCRIPTION       object
DOB_VIOLATION_NUMBER        object
BBL                         object
BIN                         object
BORO                         int64
BLOCK                      float64
LOT                        float64
HEARING_DATE                 int64
HEARING_TIME                 int64
SERVED_DATE                  int64
ISSUE_DATE                   int64
SEVERITY                    object
VIOLATION_TYPE              object
RESPONDENT_NAME             object
RESPONDENT_HOUSE_NUMBER     object
RESPONDENT_STREET           object
RESPONDENT_CITY             object
RESPONDENT_ZIP              object
PENALITY_IMPOSED            object
AMOUNT_PAID                 object
BALANCE_DUE                 object
AGGRAVATED_LEVEL            object
HEARING_STATUS              object
CERTIFICATION_STATUS        object
dtype: object

Convert Date

In [58]:
pd.set_option('chained_assignment', None)
de2['Served Date'] = pd.to_datetime(de2['SERVED_DATE'].astype(str), format='%Y%m%d', errors='coerce')

de2['Year'] = de2['Served Date'].dt.year

Insert lat/lon from pluto

In [59]:
pathfdny = "C:\\Users\\MehriD01\\OneDrive - New York City Housing Authority\\Documents\\FDNY\\"

dp = pd.read_csv(pathfdny + "pluto_25v4.csv", low_memory=False)

# Borough mapping
boro_map = {
    'MN': '1',
    'BX': '2',
    'BK': '3',
    'QN': '4',
    'SI': '5'
}

# Create BBL
dp['BBL'] = (
    dp['borough'].map(boro_map) +
    dp['block'].astype(int).astype(str).str.zfill(5) +
    dp['lot'].astype(int).astype(str).str.zfill(4)
)

dp.BBL = dp.BBL.astype(str)

In [60]:
dp.cd

0         411.0
1         411.0
2         411.0
3         411.0
4         411.0
          ...  
858639    204.0
858640    402.0
858641    305.0
858642    307.0
858643    502.0
Name: cd, Length: 858644, dtype: float64

Merge

In [61]:
dp = dp[["BBL", "latitude", "longitude", "yearbuilt", "bldgclass", "cd"]]

lat = dp.set_index('BBL')['latitude'].to_dict()
de2["lat"] = de2["BBL"].map(lat)

lon = dp.set_index('BBL')['longitude'].to_dict()
de2["lon"] = de2["BBL"].map(lon)

yb = dp.set_index('BBL')['yearbuilt'].to_dict()
de2["Year Built"] = de2["BBL"].map(yb)

#building class
yb = dp.set_index('BBL')['bldgclass'].to_dict()
de2["Building Class"] = de2["BBL"].map(yb)

yb = dp.set_index('BBL')['cd'].to_dict()
de2["Community District"] = de2["BBL"].map(yb)
de2["Community District"] = de2["Community District"].fillna(0).astype(int)
de2["Community District"] = de2["Community District"].astype(str)

In [62]:
#de2.to_csv(pathdob + "test.csv", index=False)

Get centroid of the community district

In [63]:
import geopandas as gpd

pathshape = "C:\\Users\\MehriD01\\OneDrive - New York City Housing Authority\\Documents\\FDNY\\nycd_26a\\"

# Load shapefile
gdf = gpd.read_file(pathshape + "nycd.shp")

# project to a projected CRS
gdf_proj = gdf.to_crs(epsg=2263)

# calculate centroid there
gdf_proj["centroid"] = gdf_proj.geometry.centroid

# convert centroid back to lat/lon
centroids = gdf_proj.set_geometry("centroid").to_crs(epsg=4326)

gdf["lon"] = centroids.geometry.x
gdf["lat"] = centroids.geometry.y

#gdf = gdf[["BoroCD", "lon", "lat"]]

gdf["BoroCD"] = gdf["BoroCD"].astype(str)

Merge

In [64]:
lat = gdf.set_index('BoroCD')['lat'].to_dict()
de2["CDlat"] = de2["Community District"].map(lat)

lon = gdf.set_index('BoroCD')['lon'].to_dict()
de2["CDlon"] = de2["Community District"].map(lon)

Create Decade Label

In [65]:
#de2['Decade'] = ((de2['Year'] // 10) * 10).astype(str) + 's'

In [66]:
de2['Year'] = de2['Year'].fillna(0).astype(int)

In [67]:
bins = [1989, 1999, 2009, 2019, 2026]
labels = ['1990-1999', '2000-2009', '2010-2019', '2020-2026']

de2['Decade'] = pd.cut(de2['Year'], bins=bins, labels=labels)

de2.loc[de2['Year'] == 0, 'Decade'] = pd.NA

Rename

In [68]:
de2['Decade'] = de2['Decade'].str.replace('1990-1999', '1990s')
de2['Decade'] = de2['Decade'].str.replace('2000-2009', '2000s')
de2['Decade'] = de2['Decade'].str.replace('2010-2019', '2010s')
de2['Decade'] = de2['Decade'].str.replace('2020-2026', '2020s')


In [69]:
de2['Decade'].value_counts(dropna=False)

Decade
2000s    6535
2010s    5709
1990s    4343
2020s    3815
NaN        98
Name: count, dtype: int64

Borough codes to borough names

In [70]:
#boro  = set(de2.BORO.tolist() )

# mapping dictionary
boro_map = {
    1: 'Manhattan',
    2: 'Bronx',
    3: 'Brooklyn',
    4: 'Queens',
    5: 'Staten Island'
}

# create new column
de2['Borough'] = de2['BORO'].map(boro_map)


Building Class

In [71]:
de2["Building Class Code"] = de2["Building Class"]

bc = de2["Building Class Code"].tolist()

In [72]:
#import building classes
dc = pd.read_csv(pathdob + "Building Classifications.csv", low_memory=False)

dc['Type'] = dc['Type'].str.replace('DWELLINGS', 'DWELLING')
dc['Type'] = dc['Type'].str.replace('APARTMENTS', 'APARTMENT')
dc['Type'] = dc['Type'].str.replace('WAREHOUSES', 'WAREHOUSE')
dc['Type'] = dc['Type'].str.replace('BUILDINGS', 'BUILDING')

dc['Type'] = dc['Type'].str.replace('GARAGES', 'GARAGE')
dc['Type'] = dc['Type'].str.replace('HOTELS', 'HOTEL')
dc['Type'] = dc['Type'].str.replace('HOSPITALS AND HEALTH FACILITIES', 'HOSPITAL OR HEALTH FACILITY')
dc['Type'] = dc['Type'].str.replace('THEATRES', 'THEATER')

dc['Type'] = dc['Type'].str.replace('LOFTS', 'LOFT')
dc['Type'] = dc['Type'].str.replace('FACILITIES', 'FACILITY')
dc['Type'] = dc['Type'].str.replace('ASYLUMS AND HOMES', 'ASYLUM AND HOME')
dc['Type'] = dc['Type'].str.replace('CONDOMINIUMS', 'CONDOMINIUM')

dc['Type'] = dc['Type'].str.replace('PROPERTIES', 'PROPERTY')
dc['Type'] = dc['Type'].str.replace('DEPARTMENTS', 'DEPARTMENT')
dc['Type'] = dc['Type'].str.replace('CLASSIFICATIONS', 'CLASSIFICATION')

dc['Description'] = dc['Description'].str.title()
dc['Type'] = dc['Type'].str.title()

dc["Description and Type"] = dc["Type"] + ": " + dc['Description']

dcDic = dc.set_index('Code')['Description and Type'].to_dict()



Merge

In [75]:
de2["Building Class Description"] = de2["Building Class Code"].map(dcDic)



In [76]:
de2.to_csv(pathdob + "test.csv", index=False)

In [77]:

de2.loc[de2['Building Class'].isin([f'B{i}' for i in range(1, 10)]), 
       'Building Class'] = 'Two Family Dwellings'

de2.loc[de2['Building Class'].isin([f'C{i}' for i in range(0, 10)]), 
       'Building Class'] = 'Walk-Up Apartments'

de2.loc[de2['Building Class'].isin([f'A{i}' for i in range(0, 10)]), 
       'Building Class'] = 'One Family Dwellings'

de2.loc[de2['Building Class'].isin([f'S{i}' for i in range(0, 10)]), 
       'Building Class'] = 'Mixed Used'

de2.loc[de2['Building Class'].isin([f'D{i}' for i in range(0, 10)]), 
       'Building Class'] = 'Elevator Apartments'

de2.loc[de2['Building Class'].isin([f'K{i}' for i in range(0, 10)]), 
       'Building Class'] = 'Store Buildings'

de2.loc[de2['Building Class'].isin([f'O{i}' for i in range(0, 10)]), 
       'Building Class'] = 'Office Buildings'

de2.loc[de2['Building Class'].isin([f'R{i}' for i in range(0, 10)]), 
       'Building Class'] = 'Condominiums'

de2.loc[de2['Building Class'].isin([f'W{i}' for i in range(0, 10)]), 
       'Building Class'] = 'Educational Facilities'

de2.loc[de2['Building Class'].isin([f'V{i}' for i in range(0, 10)]), 
       'Building Class'] = 'Vacant Land'

de2.loc[de2['Building Class'].isin([f'M{i}' for i in range(0, 10)]), 
       'Building Class'] = 'Religious Facilities'

de2.loc[de2['Building Class'].isin([f'G{i}' for i in range(0, 10)]), 
       'Building Class'] = 'Garages'

de2.loc[de2['Building Class'].isin([f'F{i}' for i in range(0, 10)]), 
       'Building Class'] = 'Industrial Buildings'

de2.loc[de2['Building Class'].isin([f'E{i}' for i in range(0, 10)]), 
       'Building Class'] = 'Warehouses'

de2.loc[de2['Building Class'].isin([f'H{i}' for i in range(0, 10)]), 
       'Building Class'] = 'Hotels'

de2['Building Class'] = de2['Building Class'].str.replace('HR', 'Hotels')
de2['Building Class'] = de2['Building Class'].str.replace('HH', 'Hotels')
de2['Building Class'] = de2['Building Class'].str.replace('HS', 'Hotels')
de2['Building Class'] = de2['Building Class'].str.replace('HB', 'Hotels')
de2['Building Class'] = de2['Building Class'].str.replace('DB', 'Elevator Apartments')
de2['Building Class'] = de2['Building Class'].str.replace('Z9', 'Miscellaneous')
de2['Building Class'] = de2['Building Class'].str.replace('Z8', 'Miscellaneous')
de2['Building Class'] = de2['Building Class'].str.replace('GU', 'Garages')
de2['Building Class'] = de2['Building Class'].str.replace('GW', 'Garages')
de2['Building Class'] = de2['Building Class'].str.replace('J6', 'Theaters')
de2['Building Class'] = de2['Building Class'].str.replace('U6', 'Utilities')


de2['Building Class'] = de2['Building Class'].str.replace('Q2', 'Recreational Facilities')

r_classes = ['RA','RB','RG','RH','RK','RP','RR','RS','RT','RW', 'RX', 'RC', 'RZ', 'RI', 'RD', 'RM']
de2.loc[de2['Building Class'].isin(r_classes), 'Building Class'] = 'Condominiums'

r_classes = ['N1','N2','N3','N4','N9']
de2.loc[de2['Building Class'].isin(r_classes), 'Building Class'] = 'Asylums and Homes'

r_classes = ['I1','I2','I3','I4','I5', 'I7','I9']
de2.loc[de2['Building Class'].isin(r_classes), 'Building Class'] = 'Hospitals'

r_classes = ['P1','P2','P3','P4','P5','P7','P8','P9']
de2.loc[de2['Building Class'].isin(r_classes), 'Building Class'] = 'Public Assembly Buildings'


Building Class and Borough

In [80]:
de2["Building Class & Borough"] = de2['Building Class'] + ", " + de2.Borough
de2["Building Class Description & Borough"] = de2['Building Class Description'] + ", " + de2.Borough


Building Age Category

In [101]:
def age_category(year):
    if pd.isna(year):
        return 'Unknown'
    elif year >= 2000:
        return 'Post 2000'
    elif year >= 1980:
        return '1980 to 1999'
    elif year >= 1960:
        return '1960 to 1979'
    elif year >= 1940:
        return '1940 to 1959'
    elif year >= 1920:
        return '1920 to 1939'
    elif year >= 1900:
        return '1900 to 1919'
    else:
        return 'Pre 1900'

de2['Building Age Category'] = de2['Year Built'].apply(age_category)

Building Age and Borough

In [103]:
de2["Building Age & Borough"] = de2['Building Age Category'] + ", " + de2.Borough

Neighborhood

In [89]:
dn = pd.read_csv(pathfdny + "New_York_City_Population_By_Community_Districts_20260414.csv")
dn["CD Number"] = dn["CD Number"].astype(str)
dn["CD Number"] = dn["CD Number"].str.split(".").str[0]
dn = dn[dn["CD Number"] != 'nan']

dn['Borough2'] = dn['Borough']

dn['Borough2'] = dn['Borough2'].str.replace('Bronx', '2')
dn['Borough2'] = dn['Borough2'].str.replace('Manhattan', '1')
dn['Borough2'] = dn['Borough2'].str.replace('Brooklyn', '3')
dn['Borough2'] = dn['Borough2'].str.replace('Queens', '4')
dn['Borough2'] = dn['Borough2'].str.replace('Staten Island', '5')

dn['boro_cd'] = dn['Borough2'].astype(str) + dn['CD Number'].astype(str).str.zfill(2)

dn['2010 Population'] = dn['2010 Population'].str.replace(',', '')

dn['2010 Population'] = dn['2010 Population'].astype(int)


Merge

In [98]:
dnDic = dn.set_index('boro_cd')['CD Name'].to_dict()
de2["Community District Name"] = de2["Community District"].map(dnDic)
de2["Community District Name"] = de2["Community District Name"] + ", " + de2.Borough

In [99]:
de2.to_csv(pathdob + "test.csv", index=False)

In [104]:
de2["Count"] = 1
de2.to_csv(pathdob + "ECB Violations Illegal Conversions.csv", index=False)